In [1]:
import psycopg2
from psycopg2 import sql
from psycopg2.extras import execute_values, Json
import logging
import json
import html
import re
import os


In [2]:
import pandas as pd
from pathlib import Path
from collections import Counter
from datetime import datetime
import re

In [3]:
DB_PARAMS = {
    'dbname':   'thecall',
    'user':     'postgres',
    'password': 'password',
    'host':     'localhost',
    'port':     '5432'
}

TABLE_SCHEMA = 'articles'
TABLE_NAME   = 'proquestarticles_nite'

# Define the root mapping
OLD_ROOT = r'E:\Callproject\assembled'
NEW_ROOT = r'E:\Callproject\4_raw_files\set1'

# How many updates to accumulate before committing to the database
BATCH_SIZE = 500

conn = psycopg2.connect(**DB_PARAMS)
cur = conn.cursor()

In [14]:
# Updated MAGIC_BYTES with all remaining signatures
MAGIC_BYTES = {
    'jpg':     (b'\xff\xd8\xff',),
    'tif':     (b'\x49\x49\x2a\x00', b'\x4d\x4d\x00\x2a'),
    'png':     (b'\x89\x50\x4e\x47',),
    'indd':    (b'\x06\x06\xed\xf5\xd8\x1d\x46\xe5', b'ADOBE INDESIGN P'),
    'qxp':     (b'\x00\x00MMXPR3',),
    'gif':     (b'GIF87a', b'GIF89a'),
    'pdf':     (b'%PDF',),
    'gz':      (b'\x1f\x8b',),
    'eps':     (b'%!PS',),
    'doc':     (b'\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1',),
    'docx':    (b'\x50\x4b\x03\x04',),
    'psd':     (b'8BPS',),
    'bmp':     (b'BM',),
    'rtf':     (b'{\\rtf',),
    'ttf':     (b'\x00\x01\x00\x00\x00',),
    'sit':     (b'SIT!',),
    'mp3':     (b'\xff\xfc', b'\xff\xfb', b'\xff\xf3', b'ID3'),
    'qxtags':  (b'<v1.', b'<V1.'),
    'eps': (b'%!PS', b'\xc5\xd0\xd3\xc6'),  # plain EPS + binary EPS with preview
    'tga':  (b'\x00\x00\x03\x00', b'\x00\x00\x02\x00'),  # Targa image (uncompressed)
    'otf':  (b'OTTO',),                                    # OpenType font
    'wpd':  (b'\xffWPC',),                                 # WordPerfect
    'exe':  (b'MZ',),                                      # Windows executable
    'swf':  (b'CWS', b'FWS'),                             # Flash
    'wmv':  (b'\x30\x26\xb2\x75',),                       # Windows Media
    '7z':   (b'7z\xbc\xaf',),                             # 7-Zip archive
    'otf':  (b'OTTO',),                                    # OpenType
    'pfb':  (b'\x80\x01',),                               # Type 1 font binary
    'dmg':  (b'x\xda',),                                  # Apple disk image (zlib compressed)
}

def is_plain_text(filepath):
    """Return True if the first 512 bytes are all printable ASCII or common whitespace."""
    try:
        with open(filepath, 'rb') as f:
            chunk = f.read(512)
        if not chunk:
            return False
        return all(b >= 0x20 or b in (0x09, 0x0a, 0x0d) for b in chunk)
    except OSError:
        return False

def detect_filetype(filepath):
    try:
        with open(filepath, 'rb') as f:
            header = f.read(8)
        for filetype, signatures in MAGIC_BYTES.items():
            for sig in signatures:
                if header.startswith(sig):
                    return filetype
    except (FileNotFoundError, PermissionError, OSError):
        return None
    if is_plain_text(filepath):
        return 'txt'
    return None

In [15]:
def flush_updates(updates):
    """Write a batch of updates to the database and commit."""
    if not updates:
        return
    execute_values(
        cur,
        """
        UPDATE articles.filelist AS f
        SET filetype = v.filetype
        FROM (VALUES %s) AS v(filetype, id)
        WHERE f.id = v.id;
        """,
        updates,
        template="(%s, %s)"
    )
    conn.commit()

In [16]:
# Final pass — same batching and progress reporting as before
cur.execute("""
    SELECT id, folder, filename
    FROM articles.filelist
    WHERE filetype IS NULL;
""")
null_rows = cur.fetchall()
print(f"Rows to check: {len(null_rows)}")

total         = len(null_rows)
updates       = []
total_updated = 0
still_unknown = 0
empty_files   = 0
type_counts   = Counter()
start_time    = datetime.now()

for i, (row_id, folder, filename) in enumerate(null_rows, start=1):
    remapped_folder = folder.replace(OLD_ROOT, NEW_ROOT)
    filepath = os.path.join(remapped_folder, filename)

    # Check for empty files
    try:
        if os.path.getsize(filepath) == 0:
            updates.append(('empty', row_id))
            type_counts['empty'] += 1
            continue
    except OSError:
        pass

    detected = detect_filetype(filepath)
    if detected:
        updates.append((detected, row_id))
        type_counts[detected] += 1
    else:
        still_unknown += 1

    if len(updates) >= BATCH_SIZE:
        flush_updates(updates)
        total_updated += len(updates)
        updates = []

    if i % 500 == 0 or i == total:
        elapsed   = (datetime.now() - start_time).total_seconds()
        rate      = i / elapsed if elapsed > 0 else 0
        remaining = (total - i) / rate if rate > 0 else 0
        print(
            f"[{datetime.now().strftime('%H:%M:%S')}] "
            f"{i:>6,} / {total:,} ({i/total*100:5.1f}%) | "
            f"committed: {total_updated:,} | "
            f"still unknown: {still_unknown:,} | "
            f"~{remaining/60:.1f} min remaining"
        )

flush_updates(updates)
total_updated += len(updates)

print("\n--- Final pass complete ---")
print(f"Newly identified: {total_updated:,}")
print(f"Still unknown:    {still_unknown:,}")
print(f"Breakdown:        {dict(type_counts)}")

Rows to check: 890
[19:55:25]    500 / 890 ( 56.2%) | committed: 0 | still unknown: 491 | ~0.0 min remaining
[19:55:25]    890 / 890 (100.0%) | committed: 0 | still unknown: 867 | ~0.0 min remaining

--- Final pass complete ---
Newly identified: 23
Still unknown:    867
Breakdown:        {'wmv': 1, 'exe': 3, '7z': 1, 'tga': 6, 'pfb': 2, 'swf': 1, 'otf': 5, 'dmg': 1, 'wpd': 3}


In [ ]:
# Mark the 3 missing files and anything still unidentified
cur.execute("""
    UPDATE articles.filelist
    SET filetype = 'missing'
    WHERE filetype IS NULL
      AND NOT EXISTS (
          SELECT 1 FROM articles.filelist f2
          WHERE f2.id = articles.filelist.id
      );
""")

# Mark whatever remains after the detection pass
cur.execute("""
    UPDATE articles.filelist
    SET filetype = 'unknown'
    WHERE filetype IS NULL;
""")
conn.commit()
print(f"Marked unknown: {cur.rowcount:,}")

In [ ]:
# Diagnostic: inspect the raw header bytes of unrecognised files
import binascii

sample_limit = 20
sample_count = 0

for row_id, folder, filename in rows:
    remapped_folder = folder.replace(OLD_ROOT, NEW_ROOT)
    filepath = os.path.join(remapped_folder, filename)

    if not os.path.exists(filepath):
        continue

    detected = detect_filetype(filepath)
    if detected is None:
        try:
            with open(filepath, 'rb') as f:
                header = f.read(16)
            print(f"{filename}")
            print(f"  hex:   {binascii.hexlify(header).decode()}")
            print(f"  ascii: {header}")
            print()
        except OSError as e:
            print(f"  ERROR: {e}")

        sample_count += 1
        if sample_count >= sample_limit:
            break

In [ ]:
# What extensions appear in the unrecognised filenames?
from collections import Counter
import os

cur.execute("""
    SELECT filename
    FROM articles.filelist
    WHERE filetype IS NULL;
""")
unrecognised_rows = cur.fetchall()

ext_counts = Counter()
no_ext = 0

for (filename,) in unrecognised_rows:
    ext = os.path.splitext(filename)[1].lower()
    if ext:
        ext_counts[ext] += 1
    else:
        no_ext += 1

print(f"Files with no extension: {no_ext}")
print("\nTop extensions found:")
for ext, count in ext_counts.most_common(30):
    print(f"  {ext:<15} {count:,}")

In [ ]:
# Sample hex headers for files with no extension and the dot-name pattern
import binascii

cur.execute("""
    SELECT id, folder, filename
    FROM articles.filelist
    WHERE filetype IS NULL;
""")
rows2 = cur.fetchall()

no_ext_sample = 0
dot_name_sample = 0

for row_id, folder, filename in rows2:
    ext = os.path.splitext(filename)[1].lower()
    is_no_ext = ext == ''
    is_dot_name = len(ext) > 2 and ' ' in ext  # space in "extension" = dot-name pattern

    if not (is_no_ext or is_dot_name):
        continue
    if is_no_ext and no_ext_sample >= 5:
        continue
    if is_dot_name and dot_name_sample >= 5:
        continue

    remapped_folder = folder.replace(OLD_ROOT, NEW_ROOT)
    filepath = os.path.join(remapped_folder, filename)

    if not os.path.exists(filepath):
        continue

    try:
        with open(filepath, 'rb') as f:
            header = f.read(16)
        label = 'NO-EXT' if is_no_ext else 'DOT-NAME'
        print(f"[{label}] {filename}")
        print(f"  hex:   {binascii.hexlify(header).decode()}")
        print(f"  ascii: {header}")
        print()
        if is_no_ext:
            no_ext_sample += 1
        else:
            dot_name_sample += 1
    except OSError:
        pass

    if no_ext_sample >= 5 and dot_name_sample >= 5:
        break

In [ ]:
# Mark macOS resource forks (._filename pattern)
cur.execute(r"""
    UPDATE articles.filelist
    SET filetype = 'resource_fork'
    WHERE filetype IS NULL
      AND filename LIKE '.\_%';
""")
print(f"Resource forks marked: {cur.rowcount}")

# Mark macOS Icon files
cur.execute("""
    UPDATE articles.filelist
    SET filetype = 'mac_system'
    WHERE filetype IS NULL
      AND filename IN ('Icon', 'Icon\r');
""")
print(f"Icon files marked: {cur.rowcount}")

# Mark Mac Desktop DB files
cur.execute("""
    UPDATE articles.filelist
    SET filetype = 'mac_system'
    WHERE filetype IS NULL
      AND filename IN ('Desktop DB', 'Desktop DF');
""")
print(f"Desktop DB files marked: {cur.rowcount}")

conn.commit()

In [ ]:
# Correct syntax — use r'' to avoid the escape sequence warning
cur.execute(r"""
    UPDATE articles.filelist
    SET filetype = 'resource_fork'
    WHERE filetype IS NULL
      AND filename LIKE '.\_%';
""")

In [ ]:
system_extensions = {
    '.ds_store':  'mac_system',
    '.idlk':      'mac_system',   # InDesign lock files
    '.snm':       'mac_system',   # Netscape mail index
    '.suit':      'mac_system',   # Mac font suitcase
    '.sea':       'mac_system',   # Mac self-extracting archive
    '.afm':       'font',         # Adobe Font Metrics
    '.t1':        'font',         # Type 1 font
    '.ttf':       'font',
    '.lst':       'txt',          # plain text list
    '.txt':       'txt',
    '.html':      'html',
    '.rtf':       'rtf',
    '.scr':       'scr',          # screen saver / script
    '.dat':       'dat',
    '.c':         'txt',          # C source
    '.1':         'txt',          # man page
    '.5':         'txt',
}

for ext, label in system_extensions.items():
    cur.execute("""
        UPDATE articles.filelist
        SET filetype = %s
        WHERE filetype IS NULL
          AND lower(filename) LIKE %s;
    """, (label, f'%{ext}'))
    print(f"{ext:<20} → {label:<15} {cur.rowcount:,} rows")

conn.commit()

In [ ]:
import binascii

cur.execute("""
    SELECT id, folder, filename
    FROM articles.filelist
    WHERE filetype IS NULL
      AND filename NOT LIKE '%.%';
""")
no_ext_rows = cur.fetchall()

seen_headers = set()
sample_count = 0

for row_id, folder, filename in no_ext_rows:
    remapped_folder = folder.replace(OLD_ROOT, NEW_ROOT)
    filepath = os.path.join(remapped_folder, filename)
    if not os.path.exists(filepath):
        continue
    try:
        with open(filepath, 'rb') as f:
            header = f.read(16)
        hex_prefix = binascii.hexlify(header[:4]).decode()
        if hex_prefix in seen_headers:
            continue
        seen_headers.add(hex_prefix)
        print(f"{filename}")
        print(f"  hex:   {binascii.hexlify(header).decode()}")
        print(f"  ascii: {header}")
        print()
        sample_count += 1
    except OSError:
        pass
    if sample_count >= 20:
        break

In [ ]:
name_patterns = [
    # pattern                           filetype
    ('DesktopPrinters DB',              'mac_system'),
    ('PkgInfo',                         'mac_system'),
    ('praying hands',                   None),          # all-zero, corrupted/empty
]

# Resource forks starting with _ (not ._) — missed by earlier filter
cur.execute(r"""
    UPDATE articles.filelist
    SET filetype = 'resource_fork'
    WHERE filetype IS NULL
      AND filename LIKE '\_%' ESCAPE '\';
""")
print(f"Underscore resource forks: {cur.rowcount}")

# temp files — binary scratch/cache fragments
cur.execute("""
    UPDATE articles.filelist
    SET filetype = 'tmp'
    WHERE filetype IS NULL
      AND filename LIKE 'temp%';
""")
print(f"Temp files: {cur.rowcount}")

# Mac system files by exact name
cur.execute("""
    UPDATE articles.filelist
    SET filetype = 'mac_system'
    WHERE filetype IS NULL
      AND filename IN ('DesktopPrinters DB', 'PkgInfo', 'getmsg');
""")
print(f"Mac system (named): {cur.rowcount}")

conn.commit()

In [ ]:
cur.execute("""
    SELECT id, folder, filename
    FROM articles.filelist
    WHERE filetype IS NULL;
""")
target_prefix = bytes.fromhex('00020000')
matches = []
for row_id, folder, filename in cur.fetchall():
    filepath = os.path.join(folder.replace(OLD_ROOT, NEW_ROOT), filename)
    try:
        with open(filepath, 'rb') as f:
            h = f.read(4)
        if h == target_prefix:
            matches.append(filename)
    except OSError:
        pass

print(f"Files with 0002 0000 prefix: {len(matches)}")
print(matches[:10])

In [ ]:
cur.execute("""
    UPDATE articles.filelist
    SET filetype = 'unknown'
    WHERE filetype IS NULL
      AND filename IN (
          '3 col MEMBERS OF FRIENDS OF',
          '1 col. DEOTIS DEAN JR',
          '2-col May Thomas',
          'N-3COL-NAACP 2002 FULL',
          '2-col May _Thomas',
          '1 col. DEOTIS DEAN%2'
      );
""")
conn.commit()
print(f"Marked unknown: {cur.rowcount}")

In [21]:
cur.execute("""
    SELECT 
        COALESCE(filetype, 'NULL (unidentified)') AS filetype,
        COUNT(*) AS total
    FROM articles.filelist
    GROUP BY filetype
    ORDER BY total DESC;
""")
for row in cur.fetchall():
    print(f"  {row[0]:<25} {row[1]:>8,}")

  tif                        136,633
  qxp                         97,836
  indd                        77,307
  jpg                         29,462
  pdf                         28,314
  resource_fork               12,078
  eps                          9,225
  empty                        6,624
  doc                          5,955
  mac_system                   5,688
  docx                         2,664
  gz                           1,555
  qxtags                         832
  gif                            594
  png                            486
  txt                            405
  psd                            165
  font                           120
  bmp                            102
  rtf                             93
  scr                             82
  dat                             44
  iso                             35
  ttf                             20
  html                            18
  unknown                         12
  sit                              9
 

In [ ]:
cur.execute("""
    SELECT id, folder, filename
    FROM articles.filelist
    WHERE filetype IS NULL;
""")
null_rows = cur.fetchall()

exists = 0
missing = 0

for row_id, folder, filename in null_rows:
    filepath = os.path.join(folder.replace(OLD_ROOT, NEW_ROOT), filename)
    if os.path.exists(filepath):
        exists += 1
    else:
        missing += 1

print(f"NULL rows where file exists on disk: {exists:,}")
print(f"NULL rows where file is missing:     {missing:,}")

In [18]:
import binascii

cur.execute("""
    SELECT id, folder, filename
    FROM articles.filelist
    WHERE filetype IS NULL;
""")
unknown_rows = cur.fetchall()

# Collect one example filename per distinct 4-byte header
header_examples = {}

for row_id, folder, filename in unknown_rows:
    filepath = os.path.join(folder.replace(OLD_ROOT, NEW_ROOT), filename)
    try:
        with open(filepath, 'rb') as f:
            header = f.read(16)
        key = binascii.hexlify(header[:4]).decode()
        if key not in header_examples:
            header_examples[key] = (filename, header)
    except OSError:
        pass

print(f"Distinct 4-byte prefixes found: {len(header_examples)}\n")
for hex_key, (filename, header) in sorted(header_examples.items()):
    print(f"  {hex_key}  {filename}")
    print(f"           ascii: {header}")
    print()

Distinct 4-byte prefixes found: 13

  00000000  praying hands
           ascii: b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'

  00000003  Selected Sites
           ascii: b'\x00\x00\x00\x03\x10ldap.bigfoo'

  0000026c  Apple 12_ RGB Standard
           ascii: b'\x00\x00\x02lappl\x02 \x00\x00mntr'

  00004949  ADVERT~4.QXD
           ascii: b'\x00\x00IIXPR3A\x00A\x00DC\x00\x00'

  00004d4d  Headers
           ascii: b'\x00\x00MM\xd8PR3\x00A\x00ALB\x00\x00'

  00020014  TISHA HOLMAN
           ascii: b'\x00\x02\x00\x14\x00\xb9\x00%\x00\xf7\x00B\x00\x00\x00\xff'

  00041201  TRAKVALS.BIN
           ascii: b'\x00\x04\x12\x01\x0c\x00\x00\x00Times\x00\x00\x00'

  00051600  Aska.pdf
           ascii: b'\x00\x05\x16\x00\x00\x02\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'

  00202020  Gide - Commentary (Bush_Kerry)
           ascii: b'\x00          \x10\x00\x00\xe1\x95'

  00436872  1-col Bill Fletcher
           ascii: b'\x00Christmas Photo'

  46434743  Ground Controls
   

In [19]:
from collections import Counter

header_counts = Counter()

for row_id, folder, filename in unknown_rows:
    filepath = os.path.join(folder.replace(OLD_ROOT, NEW_ROOT), filename)
    try:
        with open(filepath, 'rb') as f:
            header = f.read(4)
        header_counts[binascii.hexlify(header).decode()] += 1
    except OSError:
        pass

print("Top prefixes by frequency:")
for hex_key, count in header_counts.most_common(20):
    print(f"  {hex_key}   {count:,}")

Top prefixes by frequency:
  00000000   802
  00051600   2
  00202020   2
  00004949   1
  46434743   1
  4a6f7921   1
  00041201   1
  0000026c   1
  00004d4d   1
  00000003   1
  626f6f6b   1
  00436872   1
  00020014   1


In [17]:
extension_map = {
    '.lnk':        'lnk',        # Windows shortcut
    '.wmv':        'wmv',
    '.swf':        'swf',
    '.exe':        'exe',
    '.7z':         '7z',
    '.dmg':        'dmg',
    '.pcx':        'pcx',
    '.sct':        'txt',        # Scitex CT — text-based
    '.iso':        'iso',
    '.pfm':        'font',
    '.pfb':        'font',
    '.otf':        'font',
    '.wpd':        'wpd',
    '.job':        'unknown',    # QuarkXPress print job
    '.webarchive': 'webarchive', # Safari web archive
    '.tga':        'tga',
    '.mso':        'unknown',    # Office web component
}

for ext, label in extension_map.items():
    cur.execute("""
        UPDATE articles.filelist
        SET filetype = %s
        WHERE filetype IS NULL
          AND lower(filename) LIKE %s;
    """, (label, f'%{ext}'))
    print(f"{ext:<15} → {label:<12} {cur.rowcount:,} rows")

conn.commit()

.lnk            → lnk          1 rows
.wmv            → wmv          0 rows
.swf            → swf          0 rows
.exe            → exe          0 rows
.7z             → 7z           0 rows
.dmg            → dmg          0 rows
.pcx            → pcx          1 rows
.sct            → txt          2 rows
.iso            → iso          35 rows
.pfm            → font         2 rows
.pfb            → font         0 rows
.otf            → font         0 rows
.wpd            → wpd          0 rows
.job            → unknown      5 rows
.webarchive     → webarchive   1 rows
.tga            → tga          0 rows
.mso            → unknown      1 rows


In [20]:
# Anything still NULL after all passes is genuinely unclassifiable
cur.execute("""
    UPDATE articles.filelist
    SET filetype = 'empty'
    WHERE filetype IS NULL;
""")
conn.commit()
print(f"Final unknowns marked empty: {cur.rowcount:,}")

Final unknowns marked empty: 819
